# Initial Analysis - Loading and Inspecting Dataset

In [ ]:
# Load dataset
from pathlib import Path

import pandas as pd

Path("../data/processed").mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/raw/CW1_train.csv")

# Display dataset shape & column types
print("Dataset Shape:", df.shape)
print("\nColumn Data Types:\n", df.dtypes)

In [ ]:
# Check for missing values
print("\nMissing Values per Column:\n", df.isnull().sum())

In [ ]:
# Check for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

# Visualising Data Distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select numerical columns (exclude categorical features)
numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns

# Plot histograms for numerical features
df[numerical_columns].hist(figsize=(15, 12), bins=30, edgecolor="black")
plt.suptitle("Histograms of Numerical Features", fontsize=16)
plt.show()

Skewed Distributions & Outliers:
- `price` is highly right-skewed (long tail on the right).
- `carat` also appears right-skewed, meaning we might need to apply log transformation to normalize it.
- `depth` and `table` seem to have a somewhat normal distribution but may have some concentration around specific values.

Normal Distributions:
- `outcome` appears roughly normally distributed.
- Many features (`b1` to `b10`, `a6` to `a10`) have a standardized normal-like distribution (centered around zero with symmetric spread).

Uniform Distributions:
- Some of the `a` and `b` features (`a1` to `a5`, `b1` to `b5`) appear to be uniformly distributed, which suggests they might be random or engineered features.

In [ ]:
# Select categorical columns
categorical_columns = ['cut', 'color', 'clarity']

# Plot bar charts for categorical features
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(categorical_columns):
    sns.countplot(x=df[col], ax=axes[i], palette="viridis")
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Encoding Categorical Variables

In [ ]:
# Identify categorical columns
categorical_cols = ['cut', 'color', 'clarity']  # Replace with actual categorical column names

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Detecting Outliers

In [ ]:
# Select numerical columns
numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns

# Define number of rows and columns for subplots dynamically
num_cols = 5  # Set number of columns
num_rows = int(np.ceil(len(numerical_columns) / num_cols))  # Compute required rows

# Create boxplots
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(15, num_rows * 3))

# Flatten axes array for easy iteration
axes = axes.flatten()

for i, col in enumerate(numerical_columns):
    sns.boxplot(y=df[col], ax=axes[i], color='skyblue')
    axes[i].set_title(col)

# Remove empty subplots
for i in range(len(numerical_columns), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

We will not remove outliers for now because feature meanings are unclear and no extreme skews justify immediate removal.

# Feature Selection

### Filter Methods

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardize numerical features before PCA
numerical_features = df.select_dtypes(include=['float64', 'int64']).drop(columns=['outcome'])  # Exclude target
scaler = StandardScaler()
df_scaled = scaler.fit_transform(numerical_features)

# Apply PCA
pca = PCA()
pca.fit(df_scaled)

# Explained variance plot
plt.figure(figsize=(10,6))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by PCA Components')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns

X = df_encoded.drop(columns=['outcome'])
y = df['outcome']

# Reduce features to 2D space using t-SNE
X_reduced = TSNE(n_components=2, random_state=42).fit_transform(X)

# Convert to DataFrame
df_tsne = pd.DataFrame(X_reduced, columns=['TSNE1', 'TSNE2'])
df_tsne['outcome'] = y  # Add target variable for color coding

# Plot t-SNE results
plt.figure(figsize=(10,6))
sns.scatterplot(x=df_tsne['TSNE1'], y=df_tsne['TSNE2'], hue=df_tsne['outcome'], palette="coolwarm", alpha=0.7)
plt.title("t-SNE Visualization of Features")
plt.show()

- Lack of well-defined clusters: The dataset likely doesn’t have strong natural groupings, meaning that class separation isn’t straightforward.
- Non-linear relationships: Linear dimensionality reduction (like PCA) might not be the best approach, and we might need non-linear models.

In [ ]:
# Compute Pearson correlation matrix
correlation_matrix = df_encoded.corr()

# Visualize correlation with a heatmap
plt.figure(figsize=(15, 12))
sns.heatmap(correlation_matrix, cmap="coolwarm", annot=False, fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.show()

# Show correlations of all features with 'outcome'
correlation_with_target = correlation_matrix["outcome"].sort_values(ascending=False)
print("\nFeature Correlation with 'outcome':\n", correlation_with_target)

Correlations are quite low overall, so some feature transformation or synthesis may be necessary.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

# Compute mutual information scores
mi_scores = mutual_info_regression(X, y)
mi_scores = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# Display top features based on MI
print("\nMutual Information Scores:\n", mi_scores)

Features with Zero Mutual Information (Potential Candidates for Removal):
- `a6`, `a7`, `b5`, `a9`, `cut`, `b6`, `a5`, `b9`, `carat`: These contribute no additional information to predicting outcome.

Comparison from Pearson Correlation:
- `depth` is the strongest feature in both MI & correlation, reinforcing its importance.
- `color` and `clarity` show higher MI than correlation, suggesting potential nonlinear effects.
- Some features (`a6`, `a7`, `carat`) were already weak in correlation and now have zero MI, making them strong candidates for removal.

# Wrapper Methods

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

# Define models for RFE and RFA
linear_model = Lasso(alpha=0.01)  # LASSO for linear RFE
nonlinear_model = RandomForestRegressor(n_estimators=100, random_state=42)  # Random Forest for nonlinear RFE

### RFE

In [ ]:
# Apply RFE (Linear)
rfe_linear = RFE(linear_model, n_features_to_select=10)
rfe_linear.fit(X, y)
linear_ranking = pd.Series(rfe_linear.ranking_, index=X.columns).sort_values()

# Apply RFE (Nonlinear)
rfe_nonlinear = RFE(nonlinear_model, n_features_to_select=10)
rfe_nonlinear.fit(X, y)
nonlinear_ranking = pd.Series(rfe_nonlinear.ranking_, index=X.columns).sort_values()

# Display results
print("\nRFE (Linear - Lasso) Feature Rankings:\n", linear_ranking)
print("\nRFE (Nonlinear - Random Forest) Feature Rankings:\n", nonlinear_ranking)

### RFA

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import cross_val_score

# Apply RFA (Forward Selection) - Lasso
rfa_linear = SequentialFeatureSelector(linear_model, direction="forward", n_features_to_select=10, cv=5)
rfa_linear.fit(X, y)
linear_selected = X.columns[rfa_linear.get_support()]

# Apply RFA (Forward Selection) - Random Forest
rfa_nonlinear = SequentialFeatureSelector(nonlinear_model, direction="forward", n_features_to_select=10, cv=5)
rfa_nonlinear.fit(X, y)
nonlinear_selected = X.columns[rfa_nonlinear.get_support()]

# Print selected features
print("\nRFA (Lasso) Selected Features:\n", linear_selected)
print("\nRFA (Random Forest) Selected Features:\n", nonlinear_selected)

# Evaluate model performance using cross-validation
linear_scores = cross_val_score(linear_model, X[linear_selected], y, cv=5, scoring='r2')
nonlinear_scores = cross_val_score(nonlinear_model, X[nonlinear_selected], y, cv=5, scoring='r2')

# Embedded Methods

In [ ]:
from sklearn.linear_model import LassoCV
import numpy as np
import pandas as pd

# Standardize the dataset
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply Lasso regression with cross-validation
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_scaled, y)

# Get feature importance (Lasso shrinks coefficients of less important features to zero)
lasso_importance = pd.Series(np.abs(lasso.coef_), index=X.columns).sort_values(ascending=False)

# Display top features
print("\nLasso Feature Importance:\n", lasso_importance)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train a Random Forest model
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

# Get feature importance scores
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

# Display top features
print("\nRandom Forest Feature Importance:\n", rf_importance)

# Feature Selection

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split

# Assuming df_encoded is the preprocessed dataset with target variable 'outcome'
X = df_encoded.drop(columns=['outcome'])  # Features
y = df_encoded['outcome']  # Target variable

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Selection using RFE with Lasso
lasso = LassoCV(cv=5, random_state=42).fit(X_train, y_train)
rfe_lasso = RFE(lasso, n_features_to_select=10)
rfe_lasso.fit(X_train, y_train)
selected_rfe_lasso = X.columns[rfe_lasso.support_]

# Feature Selection using RFE with Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rfe_rf = RFE(rf, n_features_to_select=10)
rfe_rf.fit(X_train, y_train)
selected_rfe_rf = X.columns[rfe_rf.support_]

# Feature Selection using LassoCV
lasso_select = SelectFromModel(lasso, prefit=True)
selected_lasso = X.columns[lasso_select.get_support()]

# Feature Selection using Random Forest Importance
rf.fit(X_train, y_train)
feature_importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
selected_rf = feature_importances.nlargest(10).index.tolist()

# Combine selected features from all methods
selected_features = list(set(selected_rfe_lasso) | set(selected_rfe_rf) | set(selected_lasso) | set(selected_rf))

# Print selected features
print("Selected Features:", selected_features)

# Final dataset with selected features
X_selected = X[selected_features]
X_selected['outcome'] = df['outcome']
X_selected.to_csv('../data/processed/selected_features.csv', index=False)

# Feature Engineering

In [ ]:
import featuretools as ft
from featuretools import EntitySet

numerical_features = df_encoded.select_dtypes(include=[np.number, bool]).columns.tolist()
if "outcome" in numerical_features: 
    numerical_features.remove("outcome")  # Ensure outcome is NOT used in interactions

# Convert boolean features to integers (True/False → 1/0)
for col in df_encoded.select_dtypes(include=['bool']).columns:
    df_encoded[col] = df_encoded[col].astype(int)

# Compute correlation with outcome
target_corr = df_encoded.corr()["outcome"].abs()
high_corr_features = target_corr[target_corr > 0.05].index.tolist()
if "outcome" in high_corr_features:
    high_corr_features.remove("outcome")

# Remove outcome from dataset before feature engineering
X_features = df_encoded[numerical_features].copy()

# Feature Engineering with Featuretools
entity_set = EntitySet(id="dataset")
entity_set.add_dataframe(dataframe_name="data", dataframe=X_features, index=df_encoded.index.name or "index")

feature_matrix, feature_defs = ft.dfs(
    entityset=entity_set,
    target_dataframe_name="data",
    agg_primitives=["mean", "sum", "min", "max", "std", "count"],  # Prevent excessive transformations
    trans_primitives=["add_numeric", "multiply_numeric", "absolute"],  # Kept essential transformations only
    max_depth=2  # Reduce complexity to avoid kernel crashes
)

# Handle missing and infinite values
feature_matrix.replace([np.inf, -np.inf], np.nan, inplace=True)
feature_matrix.fillna(feature_matrix.median(), inplace=True)

Selecting engineered features

In [ ]:
import pandas as pd
from sklearn.linear_model import LassoLarsCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.model_selection import train_test_split

# Load dataset
df_encoded = pd.read_csv('../data/processed/processed_features.csv')

# Define features and target
X = df_encoded.drop(columns=['outcome'])
y = df_encoded['outcome']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Selection using LassoLarsCV (faster than LassoCV)
lasso = LassoLarsCV(cv=5).fit(X_train, y_train)

# Select features using Lasso
lasso_select = SelectFromModel(lasso, prefit=True)
selected_lasso = X.columns[lasso_select.get_support()]

# Feature Selection using RFE with Lasso (optimized with step=10)
rfe_lasso = RFE(lasso, n_features_to_select=10, step=10)
rfe_lasso.fit(X_train, y_train)
selected_rfe_lasso = X.columns[rfe_lasso.support_]

# Feature Selection using Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Select top features using RF importance
feature_importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
selected_rf = feature_importances.nlargest(10).index.tolist()

# Select features using SelectFromModel for RF
sfm = SelectFromModel(rf, threshold="mean")  # Keeps features above mean importance
sfm.fit(X_train, y_train)
selected_sfm_rf = X.columns[sfm.get_support()]

# Combine selected features from all methods
selected_features = list(set(selected_lasso) | set(selected_rfe_lasso) | set(selected_rf) | set(selected_sfm_rf))

# Print selected features
print("Selected Features:", selected_features)

# Final dataset with selected features
X_selected = X[selected_features]
X_selected['outcome'] = df_encoded['outcome']  # Correct reference to df_encoded

In [ ]:
import numpy as np

# Compute correlation matrix for selected features
corr_matrix = X_selected.corr().abs()

# Identify upper triangle of the correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation higher than threshold (e.g., 0.8)
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

# Remove highly correlated features
X_final = X_selected.drop(columns=to_drop)

# Save processed dataset
X_final.to_csv("../data/processed/processed_features.csv", index=False)

print("Final Features after correlation filtering:", X_final.columns.tolist())